In [ ]:
from pathlib import Path
from typing import List, Dict, Tuple, Union
from collections import defaultdict
import json
import os
import copy


import numpy as np
import pandas as pd
import mne
import h5py
from tqdm.auto import tqdm

from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from helpers import compute_ceiling_splithalf, compute_ceiling_variancebased

In [ ]:
ds_dir = "${MBS_THINGS_RAW_DIR}/things_eeg1/ds003825/"
ds_dir = Path(ds_dir)

TIME_POINTS = (0.0, 0.8)
SUBJECT_IDS = list(range(1, 51))
# SUBJECT_IDS = list(range(1, 5))

ROIS = {
    "occipital": "O",
    "parietal": "P",
    "temporal": "T",
    "frontal": "F",
    "central": "C",
}

ROIS_EXTENDED = [
    "occipital",
    "parietal",
    "temporal",
    "frontal",
    "central",
    "occipital_parietal",
    "whole_brain"
]

# !ls {ds_path}

# Inspect single subject EEG data

In [ ]:
# sub = "sub-38"
# file_path = ds_path / f"{sub}_task-rsvp_continuous.set"
# # epochs = mne.read_epochs_eeglab(str(file_path))
# raw = mne.io.read_raw_eeglab(str(file_path))
# raw.info

In [ ]:
# raw

In [ ]:
# raw_data_dir = ds_path / f"{sub}/eeg"
# raw_data_dir = Path(raw_data_dir)
# df1 = pd.read_csv(raw_data_dir / f"{sub}_task-rsvp_events.csv")
# df2 = pd.read_csv(raw_data_dir / f"{sub}_task-rsvp_events.tsv", sep="\t")


In [ ]:
# df1

In [ ]:
# df2

# Procssing helper functions

In [ ]:

def preprocess_raw(raw: mne.io.BaseRaw, sid: int, l_freq: float=0.1, h_freq: float=100) -> mne.io.BaseRaw:
    """
    Replicates the EEGLAB preprocessing:
      - re-reference
      - high-pass filter at 0.1 Hz
      - low-pass filter at 100 Hz
      - downsample to 100 Hz
    """

    # rereference
    # The added channel location in the montage is slightly different from what they suggest in the MATLAB code, 
    # one need to decide either to go with the montage or manually set the locations.
    if sid in [49, 50]:
        raw.pick_channels(ch_names=raw.info.ch_names[:63])

        raw.load_data().add_reference_channels(ref_channels=['FCz'])
        # raw.set_montage("brainproducts-RNP-BA-128")
        raw.info['chs'][63]['loc'][0], raw.info['chs'][63]['loc'][1], raw.info['chs'][63]['loc'][2] = 0.0, 0.0371, 0.0874
        raw.info['chs'][63]['cal'] = raw.info['chs'][62]['cal']
        raw.info['chs'][63]['range'] = raw.info['chs'][62]['range']
        # reref_data = mne.set_eeg_reference(raw.load_data(), ref_channels=["FCz"])
    else:
        raw.load_data().add_reference_channels(ref_channels=['Cz'])
        # raw.set_montage("brainproducts-RNP-BA-128")     
        raw.info['chs'][63]['loc'][0], raw.info['chs'][63]['loc'][1], raw.info['chs'][63]['loc'][2] = 0.0, 0.0, 0.095
        raw.info['chs'][63]['cal'] = raw.info['chs'][62]['cal']
        raw.info['chs'][63]['range'] = raw.info['chs'][62]['range']
    raw.set_eeg_reference("average", projection=False)
    
    raw.notch_filter(freqs=np.arange(50, 251, 50))
    raw.filter(l_freq=l_freq, h_freq=h_freq)

    return raw

def epoch_data(raw, tmin: float=-0.2, tmax: float=0.8, fs_new=100) -> mne.Epochs:
    
    events, event_id = mne.events_from_annotations(raw)
    e1_code = event_id.get("E  1")

    if e1_code is None:
        raise ValueError("No events with type 'E  1' found in annotations")

    idx = np.where(events[:, 2] == e1_code)[0]

    epochs = mne.Epochs(raw, events, event_id={"E  1": e1_code},
                        tmin=tmin, tmax=tmax, baseline=(None, 0),
                        preload=True)

    epochs_resampled = epochs.resample(fs_new, n_jobs=16)

    return epochs_resampled


def select_channels(
    epochs, 
    channel_names: List[str]=None, 
    channel_prefixes = None
    # channel_prefixes: List[str]=['O', 'P']
) -> mne.Epochs:
    assert channel_names is not None or channel_prefixes is not None, "Either channel_names or channel_prefixes must be provided"
    if channel_prefixes is not None:
        selected_channels = []
        for prefix in channel_prefixes:
            selected_channels.extend([ch for ch in epochs.info['ch_names'] if ch.startswith(prefix)])
        selected_channels = list(set(selected_channels))  # Remove duplicates if any
    else:
        selected_channels = channel_names

    return epochs.pick_channels(selected_channels)

def load_eeg_data(dataset_path, sid, **kwargs):

    vhdr_path = os.path.join(dataset_path, 'sub-'+"{:02d}".format(sid), 'eeg', 'sub-'+"{:02d}".format(sid)+'_task-rsvp_eeg.vhdr')
    event_path = os.path.join(dataset_path, 'sub-'+"{:02d}".format(sid), 'eeg', 'sub-'+"{:02d}".format(sid)+'_task-rsvp_events.tsv')
    
    eeg_raw = mne.io.read_raw_brainvision(vhdr_path, ignore_marker_types=True, preload=True)
    eeg_preprocessed = preprocess_raw(eeg_raw, sid)
    epochs = epoch_data(eeg_preprocessed, **kwargs)
    
    # # Select only occipital and parietal channels
    # epochs = select_channels(epochs, channel_prefixes=['O', 'P'])
    
    # Filter time points
    epochs = epochs.crop(tmin=TIME_POINTS[0], tmax=TIME_POINTS[1])

    channels = epochs.info['ch_names']
    times = epochs.times
    
    annotations = pd.read_csv(event_path, sep = '\t')
    assert len(annotations) == len(epochs), "Number of annotations does not match number of epochs"
    test_annotations = annotations.loc[(annotations['teststimnumber'] > -1)]
    train_annotations = annotations.loc[(annotations['stimnumber'] > -1)]

    eeg_array = epochs.get_data(units='uV')
    train_idx = train_annotations.index.to_numpy()
    test_idx = test_annotations.index.to_numpy()

    print(f" ------- Subject {sid} -------")
    print("number of training samples:", len(train_idx))
    print("number of test samples:", len(test_idx))

    eeg_train = eeg_array[train_idx]
    eeg_test = eeg_array[test_idx]

    keep_cols = ['object', 'objectnumber', 'stim']
    train_metadata = train_annotations.loc[:, keep_cols].reset_index(drop=True)
    test_metadata   = test_annotations.loc[:, keep_cols].reset_index(drop=True)
    
    return eeg_train, train_metadata, eeg_test, test_metadata, channels, times


# Test on a single subject raw data

In [ ]:
# sid = 38

# eeg_train, train_metadata, eeg_test, test_metadata, ch_names, times = load_eeg_data(ds_path, sid)

In [ ]:
# eeg_train, train_metadata, eeg_test, test_metadata, ch_names, times

In [ ]:
# sorting_indices = np.argsort(test_metadata['stim'])
# eeg_test_sorted = eeg_test[sorting_indices]

# eeg_test_sorted = eeg_test_sorted.transpose(1, 2, 0)
# eeg_test_sorted = eeg_test_sorted.reshape(eeg_test_sorted.shape[0], eeg_test_sorted.shape[1], 200, 12)
# eeg_test_sorted.shape

In [ ]:
# ch_names

In [ ]:
# selected_channels = [ch for ch in ch_names if ch.startswith('O') or ch.startswith('P')]
# selected_indices = [ch_names.index(ch) for ch in selected_channels]
# len(selected_indices)

In [ ]:
# nc_variancebased_test = compute_ceiling_variancebased(eeg_test_sorted)
# nc_splithalf_test = compute_ceiling_splithalf(eeg_test_sorted, folds=10, seed=0)

# nc_variancebased_test.shape, nc_splithalf_test.shape

In [ ]:
# fig, axes = plt.subplots(1, 1, figsize=(12, 5))
# ax = axes

# time = np.linspace(TIME_POINTS[0], TIME_POINTS[1], nc_variancebased_test.shape[1])

# sns.lineplot(x=time, y=nc_variancebased_test.mean(0), ax=ax, label='Variance-based Test')
# sns.lineplot(x=time, y=nc_splithalf_test.mean((0, -1)), ax=ax, label='Splithalf Test')
# ax.set_xlabel('Noise Ceiling')
# ax.set_ylabel('Number of channels')

# # put a horizontal line at maximum of of all ceilings
# max_ceiling_test = max(nc_variancebased_test.mean(0).max(), nc_splithalf_test.mean((0, -1)).max())
# ax.axhline(max_ceiling_test, color='k', linestyle='--', label='Max Ceiling (Test) @ {:.2f}'.format(max_ceiling_test))

# ax.legend()

# Process and run noise ceiling computations on all subjects

In [ ]:
# subject_ids = [38]

all_subject_data = {}
errors = {}

for sid in tqdm(SUBJECT_IDS, desc="Subjects"):
    try:
        eeg_train, train_metadata, eeg_test, test_metadata, ch_names, times = load_eeg_data(ds_dir, sid)
        
        all_subject_data[sid] = {
            'eeg_train': eeg_train,
            'train_metadata': train_metadata,
            'eeg_test': eeg_test,
            'test_metadata': test_metadata,
            'ch_names': ch_names,
            'times': times
        }
        
    except Exception as e:
        all_subject_data[sid] = {
            'error': str(e)
        }
        errors[sid] = str(e)

In [ ]:
channels_by_roi = {}
channels_by_roi_masks = {}
channel_names = all_subject_data[SUBJECT_IDS[1]]['ch_names']
for roi_name in tqdm(ROIS_EXTENDED):
    if roi_name == "whole_brain":
        channels_by_roi[roi_name] = channel_names
        channels_by_roi_masks[roi_name] = np.ones_like(channel_names, dtype=bool)
    elif roi_name == "occipital_parietal":
        channels_by_roi[roi_name] = [
            ch for ch in channel_names if (ch.startswith("O") or ch.startswith("P"))
        ]
        channels_by_roi_masks[roi_name] = np.where(
            np.array([(ch.startswith("O") or ch.startswith("P")) for ch in channel_names])
        )[0]
    else:
        roi_prefix = ROIS[roi_name]
        channels_by_roi[roi_name] = [
            ch for ch in channel_names if ch.startswith(roi_prefix)
        ]
        channels_by_roi_masks[roi_name] = np.where(
            np.array([ch.startswith(roi_prefix) for ch in channel_names])
        )[0]
    
    print(f"ROI {roi_name}: {len(channels_by_roi[roi_name])} channels.")

remaining_channels = set(channel_names)
for ch_list in channels_by_roi.values():
    remaining_channels -= set(ch_list)
print(f"Channels not assigned to any ROI: {remaining_channels}")


In [ ]:
# sorted(channel_names)

In [ ]:
def get_stimulus_set(df_metadata:pd.DataFrame):
    stimulus_set = df_metadata['stim'].values
    stimulus_set = [
        stim.replace("stimuli\\", "").replace("\\","/")
        for stim in stimulus_set
    ]
    stimulus_set = np.array(stimulus_set)
    sorting_indices = np.argsort(stimulus_set)
    stimulus_set = stimulus_set[sorting_indices]
    return stimulus_set, sorting_indices


In [ ]:

all_subjects_nc = {}
train_stimulus_set, test_stimulus_set = None, None
all_subject_data_sorted = {}
for sid, data in tqdm(all_subject_data.items(), desc="Computing Noise Ceilings"):
    if 'error' in data:
        all_subjects_nc[sid] = {
            'error': data['error']
        }
        continue
    all_subject_data_sorted[sid] = copy.deepcopy(data)

    try:
        # Get stimulus filenames and ensure the consistency across subjects
        subj_train_metadata = all_subject_data_sorted[sid]['train_metadata']
        train_stimulus_set_, train_sorting_indices_ = get_stimulus_set(subj_train_metadata)
        if train_stimulus_set is None:
            train_stimulus_set, train_sorting_indices = train_stimulus_set_, train_sorting_indices_
        else:
            assert np.array_equal(train_stimulus_set, train_stimulus_set_)
            # assert np.array_equal(train_sorting_indices, train_sorting_indices_)
            
        subj_test_metadata = all_subject_data_sorted[sid]['test_metadata']
        test_stimulus_set_, test_sorting_indices_ = get_stimulus_set(subj_test_metadata)
        test_stimulus_set_ = test_stimulus_set_.reshape(-1, 12)[:,0]
        assert len(np.unique(test_stimulus_set_)) == 200, "Test stimulus set does not have 200 unique stimuli"
        if test_stimulus_set is None:
            test_stimulus_set, test_sorting_indices = test_stimulus_set_, test_sorting_indices_
        else:
            assert np.array_equal(test_stimulus_set, test_stimulus_set_)
            # assert np.array_equal(test_sorting_indices, test_sorting_indices_)
            
        # Sort recordings based on stimulus names
        train_responses = all_subject_data_sorted[sid]['eeg_train'][train_sorting_indices_]
        test_responses = all_subject_data_sorted[sid]['eeg_test'][test_sorting_indices_]

        all_subject_data_sorted[sid]['eeg_train'] = {}
        all_subject_data_sorted[sid]['eeg_test'] = {}
        nc_variancebased = {}
        nc_splithalf = {}
        for roi_name, roi_mask in tqdm(channels_by_roi_masks.items(), desc=f"Subject {sid} ROIs", leave=False):
            all_subject_data_sorted[sid]['eeg_train'][roi_name] = train_responses[:, roi_mask]
            test_responses_roi = test_responses[:, roi_mask]

            eeg_test = test_responses_roi.copy()
            eeg_test = eeg_test.transpose(1, 2, 0)
            eeg_test = eeg_test.reshape(eeg_test.shape[0], eeg_test.shape[1], 200, 12)

            nc_variancebased[roi_name] = compute_ceiling_variancebased(eeg_test)
            nc_splithalf[roi_name] = compute_ceiling_splithalf(eeg_test, folds=10, seed=0)

            eeg_test_avg = eeg_test.mean(-1)
            eeg_test_avg = eeg_test_avg.transpose(2, 0, 1)  # shape: (n_stimuli, n_channels, n_timepoints)
            all_subject_data_sorted[sid]['eeg_test'][roi_name] = eeg_test_avg
    except Exception as e:
        all_subjects_nc[sid] = {
            'error': str(e)
        }
        print(f"Error for subject {sid}: {e}")
        continue
    
    all_subjects_nc[sid] = {
        'nc_variancebased': nc_variancebased,
        'nc_splithalf': nc_splithalf
    }

In [ ]:
roi_sorting = "occipital_parietal"
avg_nc_variancebased = {sid: all_subjects_nc[sid]['nc_variancebased'][roi_sorting].mean().item() for sid in all_subjects_nc if 'error' not in all_subjects_nc[sid]}
avg_nc_splithalf = {sid: all_subjects_nc[sid]['nc_splithalf'][roi_sorting].mean().item() for sid in all_subjects_nc if 'error' not in all_subjects_nc[sid]}

# Sort in descending order
avg_nc_variancebased = dict(sorted(avg_nc_variancebased.items(), key=lambda item: item[1], reverse=True))
avg_nc_splithalf = dict(sorted(avg_nc_splithalf.items(), key=lambda item: item[1], reverse=True))

np.array_equal(sorted(list(avg_nc_variancebased.keys())[:10]), sorted(list(avg_nc_splithalf.keys())[:10]))

In [ ]:
sorted(list(avg_nc_splithalf)[:10]), sorted(list(avg_nc_variancebased)[:10])
set(list(avg_nc_splithalf)[:10]) - set(list(avg_nc_variancebased)[:10]), set(list(avg_nc_variancebased)[:10]) - set(list(avg_nc_splithalf)[:10])

In [ ]:
# avg_nc_variancebased

In [ ]:
zoom = 0.5
fig, axes = plt.subplots(8, 6, figsize=(12*zoom*6, 5*8*zoom), dpi=100)

time = np.linspace(TIME_POINTS[0], TIME_POINTS[1], all_subjects_nc[SUBJECT_IDS[1]]['nc_variancebased']['occipital_parietal'].shape[1])

idx = 0
for sid in avg_nc_variancebased.keys():
    if 'error' in all_subjects_nc[sid]:
        continue
    ax = axes.flatten()[idx]
    idx += 1
    nc_variancebased_test = all_subjects_nc[sid]['nc_variancebased']['occipital_parietal']
    nc_splithalf_test = all_subjects_nc[sid]['nc_splithalf']['occipital_parietal']

    sns.lineplot(x=time, y=nc_variancebased_test.mean(0), ax=ax, label='Variance-based Test')
    sns.lineplot(x=time, y=nc_splithalf_test.mean((0, -1)), ax=ax, label='Splithalf Test')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Noise Ceiling')
    ax.set_ylim(-20, 65)

    # put a horizontal line at maximum of of all ceilings
    max_ceiling_test = max(nc_variancebased_test.mean(0).max(), nc_splithalf_test.mean((0, -1)).max())
    ax.axhline(max_ceiling_test, color='k', linestyle='--', label='Max Ceiling (Test) @ {:.2f}'.format(max_ceiling_test))
    ax.axhline(0, color='k', linestyle='--', alpha=0.5)
    
    ax.set_title(f'Subject {sid}', fontsize=16, fontweight='bold')

    ax.legend()
    
plt.tight_layout()

In [ ]:
for roi_name in ROIS_EXTENDED:
    zoom = 0.5
    fig, axes = plt.subplots(8, 6, figsize=(12*zoom*6, 5*8*zoom), dpi=100)

    time = np.linspace(TIME_POINTS[0], TIME_POINTS[1], all_subjects_nc[SUBJECT_IDS[1]]['nc_variancebased'][roi_name].shape[1])

    idx = 0
    for sid in avg_nc_variancebased.keys():
        if 'error' in all_subjects_nc[sid]:
            continue
        ax = axes.flatten()[idx]
        idx += 1
        nc_variancebased_test = all_subjects_nc[sid]['nc_variancebased'][roi_name]
        nc_splithalf_test = all_subjects_nc[sid]['nc_splithalf'][roi_name]

        sns.lineplot(x=time, y=nc_variancebased_test.mean(0), ax=ax, label='Variance-based Test')
        sns.lineplot(x=time, y=nc_splithalf_test.mean((0, -1)), ax=ax, label='Splithalf Test')
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Noise Ceiling')
        ax.set_ylim(-20, 65)

        # put a horizontal line at maximum of of all ceilings
        max_ceiling_test = max(nc_variancebased_test.mean(0).max(), nc_splithalf_test.mean((0, -1)).max())
        ax.axhline(max_ceiling_test, color='k', linestyle='--', label='Max Ceiling (Test) @ {:.2f}'.format(max_ceiling_test))
        ax.axhline(0, color='k', linestyle='--', alpha=0.5)
        
        ax.set_title(f'Subject {sid}', fontsize=16, fontweight='bold')

        ax.legend()
        
    plt.suptitle(f'Noise Ceilings for ROI: {roi_name}', fontsize=20, fontweight='bold')
    plt.tight_layout()

In [ ]:
SELECTED_SUBJECT_IDS = list(avg_nc_variancebased.keys())[:10]
SUBJECTS = [f"sub-"+"{:02d}".format(sid) for sid in SELECTED_SUBJECT_IDS]
SELECTED_SUBJECT_IDS, SUBJECTS

In [ ]:
# all_subject_data_sorted[SELECTED_SUBJECT_IDS[0]]['train_metadata']

In [ ]:
# all_subject_data_sorted[SELECTED_SUBJECT_IDS[0]]['eeg_train'].shape
# all_subject_data_sorted[SELECTED_SUBJECT_IDS[0]]['eeg_test'].shape

In [ ]:
# train_stimulus_set

### Concatenate data

In [ ]:
processed_data = {
    "train" :
        {
            "stimulus_ids": [str(stim) for stim in train_stimulus_set],
            "neural_data": {subj: 
                {roi: all_subject_data_sorted[sid]['eeg_train'][roi] for roi in ROIS_EXTENDED}
                for sid, subj in zip(SELECTED_SUBJECT_IDS, SUBJECTS)},
        },
    "test" :
        {
            "stimulus_ids": [str(stim) for stim in test_stimulus_set],
            "neural_data": {subj: 
                {roi: all_subject_data_sorted[sid]['eeg_test'][roi] for roi in ROIS_EXTENDED}
                for sid, subj in zip(SELECTED_SUBJECT_IDS, SUBJECTS)},
        },
    "noise_ceilings": {
        subj: all_subjects_nc[sid]['nc_variancebased']
        for sid, subj in zip(SELECTED_SUBJECT_IDS, SUBJECTS)
    }
}

### Metadata

In [ ]:
metadata = {
    "desc": """
    The neural data is from the THINGS EEG 1, recorded from 50 humans.
    Top 10 subjects with highest noise ceilings are selected.
    All neural data is averaged across trials for each image.
    The neural data is concatenated across subjects and channels.
    """
}
metadata_str = json.dumps(metadata, indent=2)
metadata_str = json.dumps(metadata, indent=2).encode('utf-8')

### Save to disk

In [ ]:
data_dir = '${MBS_DATA_PREP_OUTPUT_DIR}'
filename = f'things_eeg1.h5'

data_dir = Path(data_dir)
data_path = data_dir / filename

if not data_dir.exists():
    data_dir.mkdir(parents=False, exist_ok=False)

In [ ]:
with h5py.File(data_path, 'w') as f:
    for split in ['train', 'test']:
        f.create_dataset(f"{split}/stimulus_ids", data=processed_data[split]['stimulus_ids'])

        for subj in tqdm(SUBJECTS):
            for roi in tqdm(ROIS_EXTENDED, leave=False, desc=f"Writing {split} data for {subj}"):
                f.create_dataset(f"{split}/neural_data/{subj}/{roi}", data=processed_data[split]['neural_data'][subj][roi])
                
    for subj in tqdm(SUBJECTS):
        for roi in tqdm(ROIS_EXTENDED, leave=False, desc=f"Writing noise ceilings for {subj}"):
            f.create_dataset(f"noise_ceilings/{subj}/{roi}", data=processed_data['noise_ceilings'][subj][roi])

    f.attrs['metadata'] = metadata_str
    f.attrs['rois'] = ROIS_EXTENDED
    f.attrs['subjects'] = list(SUBJECTS)
    f.attrs['splits'] = ['train', 'test']
    f.attrs['max_nc'] = 100
    f.attrs['time_points'] = all_subject_data[SUBJECT_IDS[1]]['times'].tolist()
    f.attrs['channels'] = all_subject_data[SUBJECT_IDS[1]]['ch_names']
    f.close()


### Test loading the saved data

In [ ]:
loaded_data = defaultdict(dict)
with h5py.File(data_path, 'r') as f:
    splits = f.attrs['splits']
    subjects = f.attrs['subjects']
    rois = f.attrs['rois']
    for split in splits:
        loaded_data[split]['stimulus_ids'] = f[split]['stimulus_ids'][()]
        
        loaded_data[split]['neural_data'] = {}
        for subj in subjects:
            loaded_data[split]['neural_data'][subj] = {}
            for roi in rois:
                loaded_data[split]['neural_data'][subj][roi] = f[split]['neural_data'][subj][roi][()]
                
    for subj in subjects:
        loaded_data['noise_ceilings'][subj] = {}
        for roi in rois:
            loaded_data['noise_ceilings'][subj][roi] = f['noise_ceilings'][subj][roi][()]


In [ ]:
loaded_data['test'].keys()
loaded_data['test']['neural_data'][subjects[0]]['whole_brain'].shape, loaded_data['noise_ceilings'][subjects[0]]['whole_brain'].shape

In [ ]:
loaded_data['train'].keys()
loaded_data['train']['neural_data'][subjects[0]]['whole_brain'].shape

In [ ]:
len(rois), len(subjects), len(splits)

In [ ]:
subjects